# Web scraping de lanzamientos Falcon 9 desde Wikipedia
Se usa `requests` y `BeautifulSoup` para extraer la tabla *List of Falcon 9 and Falcon Heavy launches* y generar `spacex_web_scraped.csv`.

In [ ]:
import requests, re, unicodedata, pandas as pd
from bs4 import BeautifulSoup

In [ ]:
def date_time(t):
    return [x.strip() for x in list(t.strings)][0:2]
def booster_version(t):
    out=''.join([b for i,b in enumerate(t.strings) if i%2==0][0:-1]);return out
def landing_status(t):
    return [i for i in t.strings][0]
def get_mass(t):
    mass=unicodedata.normalize('NFKD',t.text).strip()
    if mass:
        mass.find('kg');new_mass=mass[0:mass.find('kg')+2]
    else:
        new_mass=0
    return new_mass
def extract_column_from_header(row):
    if row.br:row.br.extract()
    if row.a:row.a.extract()
    if row.sup:row.sup.extract()
    name=' '.join(row.contents)
    if not name.strip().isdigit():return name.strip()

In [ ]:
static_url='https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922'
response=requests.get(static_url)
soup=BeautifulSoup(response.text,'html5lib')
print(soup.title.string)
html_tables=soup.find_all('table')
first_launch_table=html_tables[2]

In [ ]:
column_names=[]
for th in first_launch_table.find_all('th'):
    name=extract_column_from_header(th)
    if name is not None and len(name)>0:column_names.append(name)
print(column_names)

In [ ]:
launch_dict={'Flight No.':[],'Launch site':[],'Payload':[],'Payload mass':[],'Orbit':[],'Customer':[],'Launch outcome':[],
'Version Booster':[],'Booster landing':[],'Date':[],'Time':[]}
extracted_row=0
for table_number,table in enumerate(soup.find_all('table','wikitable plainrowheaders collapsible')):
    for rows in table.find_all('tr'):
        if rows.th and rows.th.string:
            flight_number=rows.th.string.strip();flag=flight_number.isdigit()
        else:flag=False
        row=rows.find_all('td')
        if flag:
            extracted_row+=1
            launch_dict['Flight No.'].append(flight_number)
            dt=date_time(row[0])
            launch_dict['Date'].append(dt[0].strip(','));launch_dict['Time'].append(dt[1])
            launch_dict['Version Booster'].append(booster_version(row[1]) or row[1].a.string)
            launch_dict['Launch site'].append(row[2].a.string)
            launch_dict['Payload'].append(row[3].a.string)
            launch_dict['Payload mass'].append(get_mass(row[4]))
            launch_dict['Orbit'].append(row[5].a.string)
            launch_dict['Customer'].append(row[6].a.string if row[6].a else row[6].text.strip())
            launch_dict['Launch outcome'].append(list(row[7].strings)[0])
            launch_dict['Booster landing'].append(landing_status(row[8]))
print('Filas extraídas:',extracted_row)

In [ ]:
df=pd.DataFrame(launch_dict)
df.to_csv('spacex_web_scraped.csv',index=False)
df.head()